# Complete Pipeline: RAG + 5 Agents (RTX 4090)

This notebook does everything:
1. Build RAG from 100,000 CodeSearchNet samples
2. Run 5-Agent Pipeline
3. Evaluate on 100 HumanEval samples
4. Save all results in current directory

**Total Time:** 2-3 hours

## Part 1: Setup & Imports

In [1]:
import os
import json
import time
import torch
import gc
import re
import subprocess
import numpy as np
from pathlib import Path
from typing import Dict, List
from datetime import datetime
from dotenv import load_dotenv
from tqdm import tqdm
import faiss
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv()
HF_TOKEN = os.getenv('HF_TOKEN')

print('✅ Imports successful')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Imports successful
CUDA: True
GPU: NVIDIA GeForce RTX 4090
Memory: 24.0 GB


## Part 2: Build RAG Store (100,000 samples)

In [2]:
print('='*80)
print('BUILDING RAG STORE')
print('='*80)

# Configuration
SAMPLES_TO_INDEX = 100000  # 100000 samples for comprehensive coverage
CHUNK_MAX_LEN = 1024
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CODE_LANGUAGE = "python"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
INDEX_DIR = Path("rag_store/codesearchnet_python")
INDEX_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

print("=" * 80)
print("RAG Knowledge Base Builder - CodeSearchNet")
print("=" * 80)

# Step 1: Load CodeSearchNet
print(f"\n[1/6] Loading CodeSearchNet ({SAMPLES_TO_INDEX} samples)...")

try:
    dataset = load_dataset(
        "sentence-transformers/codesearchnet",
        split="train",
        streaming=False,
        token=HF_TOKEN
    )

    # Rename 'comment' to 'docstring'
    if "comment" in dataset.features and "docstring" not in dataset.features:
        dataset = dataset.rename_column("comment", "docstring")

    # Take first N samples
    if len(dataset) > SAMPLES_TO_INDEX:
        dataset = dataset.select(range(SAMPLES_TO_INDEX))

    print(f"✅ Loaded {len(dataset)} samples")

except Exception as e:
    print(f"❌ Failed: {e}")
    exit(1)

# Step 2: Process and chunk
print("\n[2/6] Processing and chunking documents...")

def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r", "\n").strip()
    lines = [line for line in text.split("\n") if line.strip()]
    return "\n".join(lines)

def chunk_code(doc: Dict) -> List[Dict]:
    code = clean_text(doc.get("code", ""))
    docstring = clean_text(doc.get("docstring", ""))

    if not code:
        return []

    base = f"Docstring:\n{docstring}\n\nCode:\n{code}"
    chunks = []
    for start in range(0, len(base), CHUNK_MAX_LEN):
        chunk_text = base[start:start + CHUNK_MAX_LEN]
        chunks.append({
            "text": chunk_text,
            "metadata": {
                "func_name": doc.get("func_name", "unknown"),
                "path": doc.get("path", ""),
                "language": CODE_LANGUAGE,
            }
        })
    return chunks

documents: List[Dict] = []
for sample in tqdm(dataset, desc="Processing"):
    documents.extend(chunk_code(sample))

print(f"✅ Created {len(documents)} chunks")

# Step 3: Create embeddings
print("\n[3/6] Creating embeddings...")
texts = [doc["text"] for doc in documents]

encoder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
embeddings = encoder.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
    device=DEVICE
).astype("float32")

print(f"✅ Embeddings: {embeddings.shape}")
print(f"   Device used: {DEVICE}")

# Step 4: Build FAISS index
print("\n[4/6] Building FAISS index...")
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f"✅ Index: {index.ntotal} vectors")

index_path = INDEX_DIR / "codesearchnet_python.faiss"
faiss.write_index(index, str(index_path))

# Step 5: Persist documents
print("\n[5/6] Persisting documents...")
store_path = INDEX_DIR / "documents.jsonl"
with open(store_path, "w", encoding="utf-8") as f:
    for doc in documents:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")
print(f"✅ Saved {len(documents)} documents")

# Step 6: Create manifest
print("\n[6/6] Creating manifest...")
manifest = {
    "source": "CodeSearchNet (sentence-transformers mirror)",
    "language": CODE_LANGUAGE,
    "samples_indexed": len(dataset),
    "total_chunks": len(documents),
    "embedding_model": EMBEDDING_MODEL,
    "embedding_dimension": int(embeddings.shape[1]),
    "chunk_max_len": CHUNK_MAX_LEN,
    "index_type": "FAISS IndexFlatIP",
}

manifest_path = INDEX_DIR / "manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\n" + "=" * 80)
print("✅ RAG Knowledge Base Built Successfully!")
print(f"   Location: {INDEX_DIR}")
print(f"   Samples: {len(dataset)}")
print(f"   Chunks: {len(documents)}")
print(f"   Index size: {index_path.stat().st_size / 1024 / 1024:.1f} MB")
print("=" * 80)

BUILDING RAG STORE
Using device: cuda
GPU: NVIDIA GeForce RTX 4090
RAG Knowledge Base Builder - CodeSearchNet

[1/6] Loading CodeSearchNet (100000 samples)...
✅ Loaded 100000 samples

[2/6] Processing and chunking documents...


Processing: 100%|██████████| 100000/100000 [00:02<00:00, 48916.99it/s]


✅ Created 127264 chunks

[3/6] Creating embeddings...


Batches: 100%|██████████| 1989/1989 [01:03<00:00, 31.36it/s]


✅ Embeddings: (127264, 384)
   Device used: cuda

[4/6] Building FAISS index...
✅ Index: 127264 vectors

[5/6] Persisting documents...
✅ Saved 127264 documents

[6/6] Creating manifest...

✅ RAG Knowledge Base Built Successfully!
   Location: rag_store/codesearchnet_python
   Samples: 100000
   Chunks: 127264
   Index size: 186.4 MB


## Part 3: Create RAG Retriever

In [3]:
print('Creating RAG retriever...')

# Import required libraries
from sentence_transformers import SentenceTransformer
import faiss

# Define paths
rag_dir = Path('rag_store/codesearchnet_python')

# Load embedding model
print('Loading embedding model...')
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('✅ Embedding model loaded')

# Load FAISS index
print('Loading FAISS index...')
rag_index = faiss.read_index(str(rag_dir / 'codesearchnet_python.faiss'))
print(f'✅ FAISS index loaded: {rag_index.ntotal:,} vectors')

# Load documents
print('Loading documents...')
rag_documents = []
with open(rag_dir / 'documents.jsonl') as f:
    for line in f:
        rag_documents.append(json.loads(line))
print(f'✅ Documents loaded: {len(rag_documents):,} chunks')

# Define retrieval function
def retrieve_context(query: str, k: int = 5) -> str:
    """Retrieve top-k relevant code examples"""
    query_embedding = embedding_model.encode([query])
    query_embedding = query_embedding.astype('float32')
    faiss.normalize_L2(query_embedding)
    distances, indices = rag_index.search(query_embedding, k)
    context_parts = []
    for idx in indices[0]:
        if idx < len(rag_documents):
            context_parts.append(rag_documents[idx]['text'])
            context_parts.append('\n---\n')
    return ''.join(context_parts)

print('✅ RAG retriever ready')

# Test retrieval
print('Testing retrieval...')
test_context = retrieve_context('write a function to check if a number is prime', k=3)
print(f'✅ Test successful: {len(test_context)} chars retrieved')




Creating RAG retriever...
Loading embedding model...
✅ Embedding model loaded
Loading FAISS index...
✅ FAISS index loaded: 127,264 vectors
Loading documents...
✅ Documents loaded: 127,264 chunks
✅ RAG retriever ready
Testing retrieval...
✅ Test successful: 1591 chars retrieved


## Part 4: Load Mistral-7B

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print('Loading Mistral-7B...')
model = AutoModelForCausalLM.from_pretrained('mistralai/Mistral-7B-v0.1', torch_dtype=torch.float16, device_map='auto', token=HF_TOKEN)
tokenizer = AutoTokenizer.from_pretrained('mistralai/Mistral-7B-v0.1', token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
print(f'✅ Model loaded ({torch.cuda.memory_allocated() / 1024**3:.2f} GB)')

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Mistral-7B...


Loading checkpoint shards: 100%|██████████| 2/2 [00:21<00:00, 11.00s/it]


✅ Model loaded (13.67 GB)


## Part 5: Define 5 Agents

In [52]:
class SpecAnalyzer:
    def analyze(self, text):
        try:
            import ast
            tree = ast.parse(text)
            func = next(n for n in tree.body if isinstance(n, ast.FunctionDef))

            doc = ast.get_docstring(func) or ""
            src = text

            return {
                "function_name": func.name,
                "docstring": doc,
                "full_prompt": src,
                "success": True
            }
        except Exception:
            return {
                "function_name": "unknown",
                "docstring": "",
                "full_prompt": text,
                "success": False
            }


class Developer:
    """Few-shot code generator with proper RAG usage and real code output."""

    def __init__(self, model, tokenizer, retrieve_fn):
        self.model = model
        self.tokenizer = tokenizer
        self.retrieve = retrieve_fn

    # ==============================================================
    # MAIN GENERATION
    # ==============================================================
    def generate(self, spec, use_rag=True):
        start = time.time()

        # ==== 1. Retrieve actual code examples (not comments only) ====
        rag = ""
        if use_rag:
            try:
                raw = self.retrieve(spec["docstring"], k=2)
                rag = self._keep_real_code(raw)[:400]
            except:
                rag = ""

        # ==== 2. Extract function signature ====
        signature = self._extract_signature(spec["full_prompt"])

        # ==== 3. Clean 1-line doc ====
        short_doc = self._summarize_docstring(spec["docstring"])

        # ==== 4. Build the FEW-SHOT prompt ====
        # This is the critical part: real Python code, not comments.
        prompt = ""

        if rag:
            prompt += (
                "# Example code:\n"
                f"{rag}\n\n"
            )

        prompt += (
            f"def {signature}:\n"
            f"    \"\"\"{short_doc}\"\"\"\n"
            f"    "
        )

        # ==== 5. Generate ====
        toks = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
        toks = {k: v.to(self.model.device) for k, v in toks.items()}

        with torch.no_grad():
            out = self.model.generate(
                **toks,
                max_new_tokens=300,
                temperature=0.3,
                top_p=0.95,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

        new = out[0][toks["input_ids"].shape[1]:]
        raw_body = self.tokenizer.decode(new, skip_special_tokens=True)

        # ==== 6. Clean body ====
        body = self._clean_body(raw_body)

        # ==== 7. Final function assembly ====
        final = (
            f"def {signature}:\n"
            f"    \"\"\"{short_doc}\"\"\"\n"
            f"{body}"
        )

        return {
            "code": final.strip(),
            "time": time.time() - start,
            "rag_used": bool(rag),
            "success": True
        }

    # ==============================================================
    # HELPERS
    # ==============================================================

    def _extract_signature(self, text):
        """Extract proper signature using AST."""
        import ast
        try:
            tree = ast.parse(text)
            f = next(n for n in tree.body if isinstance(n, ast.FunctionDef))

            args = []
            for a in f.args.args:
                if a.annotation:
                    args.append(f"{a.arg}: {ast.unparse(a.annotation)}")
                else:
                    args.append(a.arg)

            args_s = ", ".join(args)

            if f.returns:
                return f"{f.name}({args_s}) -> {ast.unparse(f.returns)}"
            return f"{f.name}({args_s})"

        except:
            return "function()"

    def _summarize_docstring(self, doc):
        if not doc:
            return "Function implementation."
        return doc.strip().split("\n")[0].strip()

    def _keep_real_code(self, rag):
        """Keep real code lines from RAG (NO filtering useful logic)."""
        out = []
        for ln in rag.split("\n"):
            s = ln.strip()
            if not s:
                continue
            if s.startswith("#"):
                continue
            if "(" in s or "=" in s or ":" in s:
                out.append(ln)
        return "\n".join(out)

    def _clean_body(self, body):
        """
        Extract ONLY the first valid-looking Python block generated by the model.
        Remove trailing hallucinations, repeated docstrings, and unrelated code.
        """
    
        lines = body.split("\n")
        cleaned = []
        seen_return = False
    
        for ln in lines:
            s = ln.strip()
    
            # Skip empty model junk
            if s.startswith("```") or s.startswith("def ") or s.startswith("class "):
                break
    
            # Stop at any new docstring after the first one
            if s.startswith('"""') or s.startswith("'''"):
                break
    
            # If we already got a return, stop when a new block appears
            if seen_return and (s.startswith("#") or len(s) == 0):
                break
    
            # Track return
            if s.startswith("return"):
                seen_return = True
    
            # Keep only reasonable python lines
            if any(k in s for k in ["=", "return", "for", "if", "while", "len(", "[", "("]):
                cleaned.append("    " + s)
            else:
                # ignore unrecognized junk
                if s == "":
                    cleaned.append("    ")
                else:
                    # if it's pure garbage (doc fragments, incomplete sentences) we break
                    if not re.match(r"[A-Za-z0-9_]+\(", s):
                        break
    
        # If empty → produce pass (prevent syntax error)
        if not cleaned:
            cleaned = ["    pass"]
    
        return "\n".join(cleaned)




# =====================================================================
#                              TESTER
# =====================================================================

class Tester:
    """Tester that executes generated code safely."""

    def test(self, code):
        start = time.time()
        try:
            clean_code = code.strip()

            # Remove markdown junk
            clean_code = clean_code.replace("```python", "").replace("```", "")

            f = Path("temp_test.py")
            f.write_text("import sys\nsys.path.insert(0,'.')\n" + clean_code + "\nprint('OK')")

            r = subprocess.run(
                ["python", str(f)],
                capture_output=True,
                text=True,
                timeout=5
            )
            f.unlink()

            return {
                "passed": ("OK" in r.stdout and r.returncode == 0),
                "time": time.time() - start,
                "error": r.stderr[:200] if r.stderr else "",
                "success": True
            }

        except Exception as e:
            return {
                "passed": False,
                "time": time.time() - start,
                "error": str(e)[:200],
                "success": False
            }


# =====================================================================
#                               REVIEWER
# =====================================================================

class Reviewer:
    def review(self, code):
        score = 100
        issues = []

        # Docstring (10 pts)
        if '"""' not in code:
            score -= 10
            issues.append("No docstring")

        # Syntax (20 pts)
        try:
            compile(code, "<string>", "exec")
        except:
            score -= 20
            issues.append("Syntax error")

        # Return present (15 pts)
        if "return" not in code:
            score -= 15
            issues.append("No return/yield")

        # Comments (5 pts)
        comment_ratio = code.count("#") / max(len(code.split("\n")), 1)
        if comment_ratio < 0.05:
            score -= 5
            issues.append("Insufficient comments")

        # Function def (10 pts)
        if "def " not in code:
            score -= 10
            issues.append("No function definition")

        return {
            "score": max(0, score),
            "issues": issues,
            "success": True
        }


# =====================================================================
#                               REPAIR
# =====================================================================

class Repair:
    def __init__(self, dev: Developer):
        self.dev = dev

    def repair(self, code, error, spec):
        try:
            cleaned_error = error.split("\n")[-2].strip()[:200]

            s = spec.copy()
            s["full_prompt"] = (
                "Fix this Python code:\n\n```python\n"
                f"{code[:500]}\n"
                "```\n\n"
                f"Error: {cleaned_error}\n"
                "Return ONLY corrected Python code.\n"
            )

            r = self.dev.generate(s, use_rag=False)

            return {
                "fixed_code": r["code"],
                "time": r["time"],
                "success": r["success"]
            }

        except Exception as e:
            return {
                "fixed_code": "",
                "time": 0,
                "success": False,
                "error": str(e)[:100]
            }


print("✅ All 5 agents defined")


✅ All 5 agents defined


## Part 6: Load Dataset & Initialize

In [53]:
from datasets import load_dataset
humaneval = load_dataset('openai_humaneval', split='test').select(range(100))
print(f'✅ Loaded {len(humaneval)} samples')

spec_analyzer = SpecAnalyzer()
developer = Developer(model, tokenizer, retrieve_context)
tester = Tester()
reviewer = Reviewer()
repair = Repair(developer)
print('✅ Agents ready')

✅ Loaded 100 samples
✅ Agents ready


## Part 7: Run Pipeline

In [54]:
print('='*80)
print('RUNNING 5-AGENT PIPELINE (100 samples, detailed output for first 10)')
print('='*80)

results = []

for idx, problem in enumerate(humaneval):
    # Detailed output for first 10 samples only
    if idx < 10:
        print(f'\n{"="*80}')
        print(f'SAMPLE [{idx+1}/100]: {problem["task_id"]}')
        print(f'{"="*80}')

        # Show problem
        print(f'\n📋 PROBLEM:')
        print(f'   {problem["prompt"][:150]}...')

        # Agent 1: Spec Analyzer
        print(f'\n[AGENT 1: SPEC ANALYZER]')
        print(f'   Input: Problem text ({len(problem["prompt"])} chars)')

    spec = spec_analyzer.analyze(problem['prompt'])

    if idx < 10:
        print(f'   Output: Function="{spec["function_name"]}", Docstring={len(spec.get("docstring", ""))} chars')
        print(f'   Status: {"✅ Success" if spec["success"] else "❌ Failed"}')

        # Agent 2: Developer
        print(f'\n[AGENT 2: DEVELOPER]')
        print(f'   Input: Spec (function={spec["function_name"]})')
        print(f'   RAG: Retrieving relevant code examples...')

    dev_result = developer.generate(spec, use_rag=True)

    if not dev_result['success']:
        if idx < 10:
            print(f'   Output: ❌ Generation FAILED')
            print(f'   Error: {dev_result.get("error", "Unknown")}')
        results.append({'task_id': problem['task_id'], 'status': 'failed'})
        continue

    if idx < 10:
        print(f'   Output: Code generated ({len(dev_result["code"])} chars)')
        print(f'   RAG Used: {"✅ Yes" if dev_result["rag_used"] else "❌ No"}')
        print(f'   Time: {dev_result["time"]:.2f}s')
        print(f'   Status: ✅ Success')
    
        # Show generated code preview
        code_preview = dev_result['code'][:200].replace('\n', '\n      ')
        print(f'   Code Preview:\n      {code_preview}...')
    
        # 🔥 NEW: PRINT FULL GENERATED CODE
        print("\n===== FULL GENERATED CODE =====")
        print(dev_result["code"])
        print("===== END GENERATED CODE =====\n")

        # Agent 3: Tester
        print(f'\n[AGENT 3: TESTER]')
        print(f'   Input: Generated code ({len(dev_result["code"])} chars)')
        print(f'   Action: Executing code with test cases...')

    test_result = tester.test(dev_result['code'])

    if idx < 10:
        print(f'   Output: {"✅ PASSED" if test_result["passed"] else "❌ FAILED"}')
        if not test_result['passed']:
            print(f'   Error: {test_result.get("error", "Unknown")[:100]}')
        print(f'   Time: {test_result["time"]:.2f}s')

        # Agent 4: Reviewer
        print(f'\n[AGENT 4: REVIEWER]')
        print(f'   Input: Generated code ({len(dev_result["code"])} chars)')
        print(f'   Action: Checking code quality...')

    review_result = reviewer.review(dev_result['code'])

    if idx < 10:
        print(f'   Output: Quality Score = {review_result["score"]}/100')
        if review_result.get('issues'):
            print(f'   Issues: {", ".join(review_result["issues"])}')
        else:
            print(f'   Issues: None')
        print(f'   Status: ✅ Complete')

    # Agent 5: Repair (if needed)
    final_code = dev_result['code']
    if not test_result['passed']:
        if idx < 10:
            print(f'\n[AGENT 5: REPAIR]')
            print(f'   Input: Failed code + Error message')
            print(f'   Action: Attempting to fix code...')

        repair_result = repair.repair(dev_result['code'], test_result['error'], spec)

        if repair_result['success']:
            if idx < 10:
                print(f'   Output: Fixed code generated ({len(repair_result["fixed_code"])} chars)')
                print(f'   Time: {repair_result["time"]:.2f}s')
                print(f'   Status: ✅ REPAIRED')
            final_code = repair_result['fixed_code']
            status = 'repaired'
        else:
            if idx < 10:
                print(f'   Output: ❌ Repair FAILED')
                print(f'   Error: {repair_result.get("error", "Unknown")[:100]}')
                print(f'   Status: ❌ Failed')
            status = 'failed'
    else:
        if idx < 10:
            print(f'\n[AGENT 5: REPAIR]')
            print(f'   Status: ⏭️  SKIPPED (Test passed, no repair needed)')
        status = 'passed'

    # Final Summary for this sample (detailed for first 10)
    if idx < 10:
        print(f'\n{"─"*80}')
        print(f'FINAL RESULT:')
        print(f'   Status: {status.upper()}')
        print(f'   Quality: {review_result["score"]}/100')
        print(f'   Test: {"✅ Passed" if test_result["passed"] else "❌ Failed"}')
        print(f'   RAG: {"✅ Used" if dev_result["rag_used"] else "❌ Not used"}')
        print(f'   Total Time: {dev_result["time"]:.2f}s')
        print(f'{"─"*80}')
    else:
        # Compact output for samples 11-100
        print(f'[{idx+1}/100] {problem["task_id"]}: {status} | Q:{review_result["score"]}/100 | T:{dev_result["time"]:.2f}s')

    # Store result
    results.append({
        'task_id': problem['task_id'],
        'status': status,
        'quality_score': review_result['score'],
        'test_passed': test_result['passed'],
        'rag_used': dev_result['rag_used'],
        'generation_time': dev_result['time'],
        'final_code_length': len(final_code)
    })

print('\n' + '='*80)
print('PIPELINE COMPLETE (100 samples)')
print('='*80)




RUNNING 5-AGENT PIPELINE (100 samples, detailed output for first 10)

SAMPLE [1/100]: HumanEval/0

📋 PROBLEM:
   from typing import List


def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any t...

[AGENT 1: SPEC ANALYZER]
   Input: Problem text (348 chars)
   Output: Function="has_close_elements", Docstring=213 chars
   Status: ✅ Success

[AGENT 2: DEVELOPER]
   Input: Spec (function=has_close_elements)
   RAG: Retrieving relevant code examples...
   Output: Code generated (319 chars)
   RAG Used: ✅ Yes
   Time: 7.69s
   Status: ✅ Success
   Code Preview:
      def has_close_elements(numbers: List[float], threshold: float) -> bool:
          """Check if in given list of numbers, are any two numbers closer to each other than"""
          for i in range(len(numbers)):
         ...

===== FULL GENERATED CODE =====
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """Check if in given list of n

KeyboardInterrupt: 

## Part 8: Results & Save

In [ ]:
passed = sum(1 for r in results if r['status'] == 'passed')
repaired = sum(1 for r in results if r['status'] == 'repaired')
failed = sum(1 for r in results if r['status'] == 'failed')
avg_quality = sum(r.get('quality_score', 0) for r in results) / len(results)
avg_time = sum(r.get('generation_time', 0) for r in results) / len(results)

print(f'\nResults:')
print(f'  ✅ Passed: {passed}')
print(f'  🔧 Repaired: {repaired}')
print(f'  ❌ Failed: {failed}')
print(f'  Quality: {avg_quality:.1f}/100')
print(f'  Time: {avg_time:.2f}s')
print(f'  Success: {(passed + repaired) / len(results) * 100:.1f}%')

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'phase2_complete_rtx4090_{timestamp}.json'

output_data = {'timestamp': datetime.now().isoformat(), 'phase': 'Phase 2 Complete', 'hardware': 'RTX 4090', 'rag_samples': SAMPLES_TO_INDEX, 'rag_chunks': len(documents), 'model': 'Mistral-7B-v0.1', 'samples': len(results), 'passed': passed, 'repaired': repaired, 'failed': failed, 'avg_quality_score': avg_quality, 'avg_generation_time': avg_time, 'results': results}

with open(output_file, 'w') as f:
    json.dump(output_data, f, indent=2)

print(f'\n✅ Saved: {output_file}')
print('\n' + '='*80)
print('ALL DONE!')
print('='*80)

In [ ]:
print('='*80)
print('PHASE 1 vs PHASE 2 COMPARISON WITH EVALUATION METRICS')
print('='*80)

# Load Phase 1 results
phase1_file = "results/evaluations/analysis_report_1762102266.json"

try:
    with open(phase1_file) as f:
        phase1_data = json.load(f)
    
    phase1_mistral_cot = phase1_data["evaluation_report"]["strategy_comparisons"]["chain_of_thought"]["mistral"]
    print('\n✅ Phase 1 results loaded')
except Exception as e:
    print(f'❌ Error loading Phase 1 results: {e}')
    phase1_mistral_cot = None

# Calculate Phase 2 metrics
passed = sum(1 for r in results if r['status'] == 'passed')
repaired = sum(1 for r in results if r['status'] == 'repaired')
failed = sum(1 for r in results if r['status'] == 'failed')
total_samples = len(results)

avg_quality = sum(r.get('quality_score', 0) for r in results) / total_samples if total_samples > 0 else 0
avg_time = sum(r.get('generation_time', 0) for r in results) / total_samples if total_samples > 0 else 0
rag_used_count = sum(1 for r in results if r.get('rag_used', False))

# Success rate (passed + repaired)
success_rate_phase2 = ((passed + repaired) / total_samples * 100) if total_samples > 0 else 0

print('\n' + '='*80)
print('DETAILED COMPARISON')
print('='*80)

# Table 1: Basic Metrics
print('\n📊 TABLE 1: BASIC METRICS')
print('-'*80)
print(f'{"Metric":<30} {"Phase 1 (No RAG)":<25} {"Phase 2 (With RAG)":<25}')
print('-'*80)

if phase1_mistral_cot:
    print(f'{"Model":<30} {"Mistral-7B-v0.1":<25} {"Mistral-7B-v0.1":<25}')
    print(f'{"Strategy":<30} {"Chain-of-Thought":<25} {"CoT + RAG + 5-Agents":<25}')
    print(f'{"Samples":<30} {phase1_mistral_cot["total_attempts"]:<25} {total_samples:<25}')
    print(f'{"Success Rate":<30} {phase1_mistral_cot["success_rate"]:.1f}%{"":<20} {success_rate_phase2:.1f}%')
    print(f'{"Quality Score":<30} {phase1_mistral_cot["avg_quality_score"]:.2f}{"":<20} {avg_quality:.2f}')
    print(f'{"Avg Generation Time":<30} {phase1_mistral_cot["avg_generation_time"]:.2f}s{"":<20} {avg_time:.2f}s')
    print(f'{"Output Length (chars)":<30} {phase1_mistral_cot["avg_output_length"]:.0f}{"":<20} {sum(r.get("final_code_length", 0) for r in results) / total_samples:.0f}')

# Table 2: Phase 2 Specific Metrics
print('\n📊 TABLE 2: PHASE 2 ENHANCEMENTS')
print('-'*80)
print(f'{"Metric":<40} {"Value":<20} {"Percentage":<20}')
print('-'*80)
print(f'{"Passed (First Try)":<40} {passed:<20} {passed/total_samples*100:.1f}%')
print(f'{"Repaired (Fixed by Agent)":<40} {repaired:<20} {repaired/total_samples*100:.1f}%')
print(f'{"Failed (Unfixable)":<40} {failed:<20} {failed/total_samples*100:.1f}%')
print(f'{"RAG Context Used":<40} {rag_used_count:<20} {rag_used_count/total_samples*100:.1f}%')
print(f'{"Total Success (Passed+Repaired)":<40} {passed+repaired:<20} {success_rate_phase2:.1f}%')

# Table 3: Improvement Analysis
if phase1_mistral_cot:
    print('\n📊 TABLE 3: IMPROVEMENT ANALYSIS')
    print('-'*80)
    print(f'{"Metric":<40} {"Improvement":<20} {"Change %":<20}')
    print('-'*80)
    
    quality_improvement = avg_quality - phase1_mistral_cot["avg_quality_score"]
    time_increase = avg_time - phase1_mistral_cot["avg_generation_time"]
    success_improvement = success_rate_phase2 - phase1_mistral_cot["success_rate"]
    
    print(f'{"Quality Score":<40} {quality_improvement:+.2f}{"":<15} {quality_improvement/phase1_mistral_cot["avg_quality_score"]*100:+.1f}%')
    print(f'{"Generation Time":<40} {time_increase:+.2f}s{"":<15} {time_increase/phase1_mistral_cot["avg_generation_time"]*100:+.1f}%')
    print(f'{"Success Rate":<40} {success_improvement:+.1f}%{"":<15} {success_improvement/phase1_mistral_cot["success_rate"]*100:+.1f}%')

# Table 4: Quality Distribution
print('\n📊 TABLE 4: QUALITY SCORE DISTRIBUTION')
print('-'*80)
print(f'{"Quality Range":<30} {"Count":<15} {"Percentage":<15}')
print('-'*80)

quality_ranges = {
    '90-100 (Excellent)': (90, 100),
    '80-89 (Good)': (80, 89),
    '70-79 (Fair)': (70, 79),
    '60-69 (Poor)': (60, 69),
    '0-59 (Very Poor)': (0, 59)
}

for range_name, (min_q, max_q) in quality_ranges.items():
    count = sum(1 for r in results if min_q <= r.get('quality_score', 0) <= max_q)
    percentage = count / total_samples * 100 if total_samples > 0 else 0
    print(f'{range_name:<30} {count:<15} {percentage:.1f}%')

# Table 5: Time Performance Analysis
print('\n📊 TABLE 5: TIME PERFORMANCE ANALYSIS')
print('-'*80)
print(f'{"Metric":<40} {"Value":<20}')
print('-'*80)

times = [r.get('generation_time', 0) for r in results if r.get('generation_time', 0) > 0]
if times:
    print(f'{"Min Generation Time":<40} {min(times):.2f}s')
    print(f'{"Max Generation Time":<40} {max(times):.2f}s')
    print(f'{"Avg Generation Time":<40} {avg_time:.2f}s')
    print(f'{"Median Generation Time":<40} {sorted(times)[len(times)//2]:.2f}s')
    print(f'{"Total Time (100 samples)":<40} {sum(times):.2f}s ({sum(times)/60:.1f} min)')

# Table 6: ROI Analysis
if phase1_mistral_cot:
    print('\n📊 TABLE 6: ROI (Return on Investment) ANALYSIS')
    print('-'*80)
    print(f'{"Metric":<40} {"Value":<20}')
    print('-'*80)
    
    quality_gain = avg_quality - phase1_mistral_cot["avg_quality_score"]
    time_cost = avg_time - phase1_mistral_cot["avg_generation_time"]
    roi = quality_gain / time_cost if time_cost > 0 else 0
    
    print(f'{"Quality Gain":<40} {quality_gain:+.2f} points')
    print(f'{"Time Cost":<40} {time_cost:+.2f}s')
    print(f'{"ROI (Quality/Time)":<40} {roi:.2f} points/sec')
    print(f'{"Repair Success Rate":<40} {repaired/(repaired+failed)*100 if (repaired+failed) > 0 else 0:.1f}%')
    print(f'{"RAG Effectiveness":<40} {"High" if rag_used_count/total_samples > 0.9 else "Medium"}')

# Summary
print('\n' + '='*80)
print('KEY FINDINGS')
print('='*80)

if phase1_mistral_cot:
    print(f'\n✅ Phase 1 (Baseline):')
    print(f'   • Quality: {phase1_mistral_cot["avg_quality_score"]:.2f}')
    print(f'   • Time: {phase1_mistral_cot["avg_generation_time"]:.2f}s')
    print(f'   • Success: {phase1_mistral_cot["success_rate"]:.1f}%')
    
    print(f'\n✅ Phase 2 (RAG + 5-Agents):')
    print(f'   • Quality: {avg_quality:.2f} ({quality_improvement:+.2f} improvement)')
    print(f'   • Time: {avg_time:.2f}s ({time_increase:+.2f}s overhead)')
    print(f'   • Success: {success_rate_phase2:.1f}% ({success_improvement:+.1f}% improvement)')
    print(f'   • Repair Rate: {repaired} samples fixed automatically')
    print(f'   • RAG Usage: {rag_used_count}/{total_samples} samples')
    
    print(f'\n📈 Overall Assessment:')
    if quality_improvement > 0:
        print(f'   ✅ Quality improved by {quality_improvement:.2f} points ({quality_improvement/phase1_mistral_cot["avg_quality_score"]*100:.1f}%)')
    else:
        print(f'   ⚠️  Quality decreased by {abs(quality_improvement):.2f} points ({quality_improvement/phase1_mistral_cot["avg_quality_score"]*100:.1f}%)')
    
    if success_improvement > 0:
        print(f'   ✅ Success rate improved by {success_improvement:.1f}%')
    else:
        print(f'   ⚠️  Success rate decreased by {abs(success_improvement):.1f}%')
    
    if repaired > 0:
        print(f'   ✅ Repair agent fixed {repaired} failed samples')
    
    print(f'   ⏱️  Time overhead: {time_increase:.2f}s per sample')
    
    if time_increase > 0:
        print(f'   💡 Trade-off: +{time_increase:.2f}s for {quality_improvement:+.2f} quality points')
    
    if roi != 0:
        print(f'   🎯 ROI: {roi:.2f} quality points per second of overhead')

print('\n' + '='*80)
